# 🧠 Train Gemma for Kernal Agent (Desktop Automation)

This notebook fine-tunes Gemma-2B to understand desktop automation commands.

**Free GPU**: Runtime → Change runtime type → T4 GPU

**Time**: ~30-60 minutes with 5000 examples

In [ ]:
# Install dependencies
!pip install -q torch transformers peft datasets accelerate bitsandbytes huggingface_hub sentencepiece

In [ ]:
# Login to HuggingFace (required for Gemma)
from huggingface_hub import login
login()  # Enter your HF token

In [ ]:
import json
import torch
from pathlib import Path
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments, 
    Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Load Training Data from HuggingFace

In [ ]:
# System prompt for our interpreter
SYSTEM_PROMPT = """You are a desktop automation interpreter. Parse user commands into structured JSON with:
- intent: The action type (OPEN_APP, CLICK_UI, TYPE_TEXT, WEB_SEARCH, FILE_OP, SYSTEM_CONTROL, MULTI_ACTION)
- entities: Extracted parameters (app, target, text, url, path, etc.)
- capabilities_needed: Required automation capabilities
- confidence: Your confidence 0.0-1.0
- risk: low|medium|high

Return ONLY valid JSON."""

def format_prompt(instruction, input_text, output_text):
    return f"""<start_of_turn>user
{instruction}

User command: {input_text}<end_of_turn>
<start_of_turn>model
{output_text}<end_of_turn>"""

In [ ]:
# Load and convert datasets
def load_glaive_function(max_examples=2000):
    """Load Glaive function calling dataset."""
    print("Loading glaive-function-calling...")
    try:
        ds = load_dataset("glaiveai/glaive-function-calling-v2", split="train", streaming=True)
        examples = []
        
        for i, ex in enumerate(ds):
            if i >= max_examples:
                break
            
            messages = ex.get("conversations", ex.get("messages", []))
            user_msg = ""
            assistant_msg = ""
            
            for msg in messages:
                role = msg.get("role", msg.get("from", ""))
                content = msg.get("content", msg.get("value", ""))
                
                if role in ["user", "human"]:
                    user_msg = content
                elif role in ["assistant", "gpt"]:
                    assistant_msg = content[:500]  # Truncate
            
            if user_msg:
                # Create structured output
                output = {
                    "intent": "FUNCTION_CALL",
                    "entities": {"query": user_msg[:100]},
                    "capabilities_needed": ["function_calling"],
                    "confidence": 0.85,
                    "risk": "low"
                }
                examples.append({
                    "instruction": SYSTEM_PROMPT,
                    "input": user_msg,
                    "output": json.dumps(output)
                })
        
        print(f"  Loaded {len(examples)} examples")
        return examples
    except Exception as e:
        print(f"  Error: {e}")
        return []


def create_desktop_automation_examples():
    """Create synthetic desktop automation examples."""
    examples = []
    
    # Template-based generation
    templates = [
        # App launching
        ("open {app}", "OPEN_APP", ["app_launcher"]),
        ("launch {app}", "OPEN_APP", ["app_launcher"]),
        ("start {app}", "OPEN_APP", ["app_launcher"]),
        ("close {app}", "CLOSE_APP", ["app_launcher"]),
        
        # Web
        ("search for {query} on google", "WEB_SEARCH", ["web_navigation"]),
        ("go to {url}", "WEB_NAVIGATE", ["web_navigation"]),
        ("open {url} in chrome", "WEB_NAVIGATE", ["app_launcher", "web_navigation"]),
        
        # UI
        ("click on {target}", "CLICK_UI", ["ui_automation"]),
        ("click the {target} button", "CLICK_UI", ["ui_automation"]),
        ("type {text}", "TYPE_TEXT", ["text_input"]),
        ("enter {text}", "TYPE_TEXT", ["text_input"]),
        
        # System
        ("turn up the volume", "VOLUME_CONTROL", ["volume_control"]),
        ("mute", "VOLUME_CONTROL", ["volume_control"]),
        ("set brightness to {value}%", "BRIGHTNESS_CONTROL", ["brightness_control"]),
        ("take a screenshot", "SCREENSHOT", ["screen_capture"]),
        
        # File
        ("create a new folder called {name}", "FILE_OP", ["file_system"]),
        ("delete {file}", "FILE_OP", ["file_system"]),
        ("move {file} to {destination}", "FILE_OP", ["file_system"]),
    ]
    
    apps = ["chrome", "notepad", "spotify", "discord", "vscode", "word", "excel", "explorer", "terminal", "settings"]
    targets = ["save", "submit", "cancel", "next", "back", "menu", "close", "minimize", "maximize"]
    queries = ["python tutorials", "weather today", "best restaurants", "coding tips", "news"]
    texts = ["hello world", "test message", "my notes", "reminder"]
    
    for template, intent, capabilities in templates:
        for app in apps[:5]:
            for target in targets[:3]:
                for query in queries[:2]:
                    cmd = template.format(
                        app=app, target=target, query=query, 
                        text=texts[0], url="google.com", 
                        name="projects", file="test.txt", 
                        destination="Documents", value="50"
                    )
                    
                    output = {
                        "intent": intent,
                        "entities": {"app": app, "target": target, "query": query},
                        "capabilities_needed": capabilities,
                        "confidence": 0.9,
                        "risk": "low"
                    }
                    
                    examples.append({
                        "instruction": SYSTEM_PROMPT,
                        "input": cmd,
                        "output": json.dumps(output)
                    })
    
    print(f"  Created {len(examples)} synthetic examples")
    return examples[:2000]  # Limit


# Combine all datasets
all_examples = []
all_examples.extend(load_glaive_function(2000))
all_examples.extend(create_desktop_automation_examples())

# Deduplicate
seen = set()
unique = []
for ex in all_examples:
    if ex["input"] not in seen:
        seen.add(ex["input"])
        unique.append(ex)

print(f"\nTotal unique examples: {len(unique)}")
dataset = Dataset.from_list(unique)

## 2. Load Gemma Model with 4-bit Quantization

In [ ]:
MODEL_NAME = "google/gemma-2b"

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Prepare Dataset

In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    prompts = []
    for i in range(len(examples["input"])):
        prompt = format_prompt(
            examples["instruction"][i],
            examples["input"][i],
            examples["output"][i]
        )
        prompts.append(prompt)
    
    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Split and tokenize
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"].map(tokenize_function, batched=True, remove_columns=split["train"].column_names)
eval_dataset = split["test"].map(tokenize_function, batched=True, remove_columns=split["test"].column_names)

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

## 4. Train!

In [ ]:
OUTPUT_DIR = "gemma-interpreter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("🚀 Starting training...")
trainer.train()

In [ ]:
# Save the model
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"✅ Model saved to {OUTPUT_DIR}/final")

## 5. Test the Model

In [ ]:
# Test commands
test_commands = [
    "open notepad",
    "search for python tutorials on google",
    "click on the save button",
    "turn up the volume",
    "create a new folder called projects",
]

model.eval()

for cmd in test_commands:
    prompt = f"""<start_of_turn>user
{SYSTEM_PROMPT}

User command: {cmd}<end_of_turn>
<start_of_turn>model
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("<start_of_turn>model")[-1].split("<end_of_turn>")[0].strip()
    
    print(f"\n{'='*50}")
    print(f"Command: {cmd}")
    print(f"Response: {response[:300]}")

## 6. Download Model

In [ ]:
# Zip and download
!zip -r gemma-interpreter.zip gemma-interpreter/

from google.colab import files
files.download('gemma-interpreter.zip')

print("\n📦 Download complete! Extract and use with Ollama or directly.")

## 7. Push to HuggingFace Hub (Optional)

In [ ]:
# Push to HuggingFace Hub
HF_USERNAME = "your-username"  # Change this
REPO_NAME = "gemma-desktop-interpreter"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"\n🚀 Model pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")